In [ ]:
from pathlib import Path

import torch

data_dir = Path("data/wikiart")
run_dir = Path("dcgan_results/run_01")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
x = torch.randn(256, 256, device=device)
print((x @ x).device)


In [ ]:
import torch
from torch import nn

latent_dim = 100
base_channels = 64


class Generator(nn.Module):
    def __init__(self, latent_dim=100, channels=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, channels * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(channels * 8),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(channels * 8, channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(channels * 4),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(channels * 4, channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(channels * 2),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(channels * 2, channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(channels, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, noise):
        return self.network(noise)


class Discriminator(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, channels, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels, channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(channels * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels * 2, channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(channels * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels * 4, channels * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(channels * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels * 8, 1, 4, 1, 0),
        )

    def forward(self, images):
        return self.network(images).flatten(1).squeeze(1)

In [ ]:
generator = Generator(latent_dim, base_channels).to(device).eval()
discriminator = Discriminator(base_channels).to(device).eval()

test_batch_size = 4

with torch.inference_mode():
    noise = torch.randn(test_batch_size, latent_dim, 1, 1, device=device)
    fake_images = generator(noise)
    logits = discriminator(fake_images)

assert fake_images.shape == (test_batch_size, 3, 64, 64)
assert logits.shape == (test_batch_size,)
assert fake_images.device.type == logits.device.type == device.type

print("Generated images:", tuple(fake_images.shape))
print("Discriminator logits:", tuple(logits.shape))
print("Device:", fake_images.device)

In [ ]:
import torch
from torch import nn

test_generator = Generator(latent_dim, base_channels).to(device)
test_discriminator = Discriminator(base_channels).to(device)

g_optimizer = torch.optim.Adam(
    test_generator.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
d_optimizer = torch.optim.Adam(
    test_discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
criterion = nn.BCEWithLogitsLoss()

batch_size = 4
test_images = torch.rand(batch_size, 3, 64, 64, device=device) * 2 - 1
noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
generated_images = test_generator(noise)

# Update the discriminator.
d_optimizer.zero_grad(set_to_none=True)

real_logits = test_discriminator(test_images)
fake_logits = test_discriminator(generated_images.detach())

d_loss = (
    criterion(real_logits, torch.ones_like(real_logits))
    + criterion(fake_logits, torch.zeros_like(fake_logits))
) / 2

assert torch.isfinite(d_loss).item(), "Discriminator loss is not finite."
d_loss.backward()
d_optimizer.step()

# Update the generator while keeping discriminator weights fixed.
g_optimizer.zero_grad(set_to_none=True)
test_discriminator.requires_grad_(False)

fake_logits = test_discriminator(generated_images)
g_loss = criterion(fake_logits, torch.ones_like(fake_logits))

assert torch.isfinite(g_loss).item(), "Generator loss is not finite."
g_loss.backward()
g_optimizer.step()
test_discriminator.requires_grad_(True)

print(f"Discriminator loss: {d_loss.item():.4f}")
print(f"Generator loss: {g_loss.item():.4f}")
print("Training device:", generated_images.device)

In [ ]:
checkpoint_path = run_dir.parent / "checkpoints" / "smoke_test.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "global_step": 1,
        "generator": test_generator.state_dict(),
        "discriminator": test_discriminator.state_dict(),
        "g_optimizer": g_optimizer.state_dict(),
        "d_optimizer": d_optimizer.state_dict(),
    },
    checkpoint_path,
)

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

restored_generator = Generator(latent_dim, base_channels).to(device)
restored_discriminator = Discriminator(base_channels).to(device)
restored_generator.load_state_dict(checkpoint["generator"])
restored_discriminator.load_state_dict(checkpoint["discriminator"])

restored_g_optimizer = torch.optim.Adam(restored_generator.parameters())
restored_d_optimizer = torch.optim.Adam(restored_discriminator.parameters())
restored_g_optimizer.load_state_dict(checkpoint["g_optimizer"])
restored_d_optimizer.load_state_dict(checkpoint["d_optimizer"])

with torch.inference_mode():
    expected_images = test_generator.eval()(noise)
    restored_images = restored_generator.eval()(noise)
    torch.testing.assert_close(restored_images, expected_images)

    expected_logits = test_discriminator.eval()(expected_images)
    restored_logits = restored_discriminator.eval()(restored_images)
    torch.testing.assert_close(restored_logits, expected_logits)

print("Checkpoint saved:", checkpoint_path)
print("Restored training step:", checkpoint["global_step"])
print("Checkpoint verification passed.")

In [ ]:
from itertools import islice

image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

if not data_dir.is_dir():
    raise FileNotFoundError(data_dir)
else:
    image_paths = (
        path for path in data_dir.rglob("*")
        if path.suffix.lower() in image_extensions and path.is_file()
    )
    examples = list(islice(image_paths, 5))

    if examples:
        print("Example image paths:")
        for path in examples:
            print(path.relative_to(data_dir))
    else:
        raise RuntimeError(f"No images found under {data_dir}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


class WikiArtDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


preview_dataset = WikiArtDataset(examples, image_transform)
preview_loader = DataLoader(preview_dataset, batch_size=4, num_workers=0)
image_batch = next(iter(preview_loader))

print("Batch shape:", tuple(image_batch.shape))
print(f"Pixel range: [{image_batch.min().item():.3f}, {image_batch.max().item():.3f}]")

fig, axes = plt.subplots(1, len(image_batch), figsize=(8, 2))
for ax, image in zip(axes, image_batch):
    display_image = (image * 0.5 + 0.5).clamp(0, 1)
    ax.imshow(display_image.permute(1, 2, 0).numpy())
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import random
import torch
from torch import nn

wikiart_paths = sorted(
    path for path in data_dir.rglob("*")
    if path.suffix.lower() in image_extensions and path.is_file()
)
print("Image files found:", len(wikiart_paths))
assert len(wikiart_paths) >= 32, "Expected at least 32 WikiArt images."

trial_paths = random.Random(42).sample(wikiart_paths, 32)
trial_dataset = WikiArtDataset(trial_paths, image_transform)
trial_loader = DataLoader(trial_dataset, batch_size=8, num_workers=0)

trial_g = Generator(latent_dim, base_channels).to(device)
trial_d = Discriminator(base_channels).to(device)
trial_g_optimizer = torch.optim.Adam(
    trial_g.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
trial_d_optimizer = torch.optim.Adam(
    trial_d.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
trial_loss_fn = nn.BCEWithLogitsLoss()

from time import perf_counter

if device.type == "cuda":
    torch.cuda.synchronize()
start_time = perf_counter()

for step, real_images in enumerate(trial_loader, start=1):
    real_images = real_images.to(device)
    noise = torch.randn(real_images.size(0), latent_dim, 1, 1, device=device)
    fake_images = trial_g(noise)

    trial_d_optimizer.zero_grad(set_to_none=True)
    real_logits = trial_d(real_images)
    fake_logits = trial_d(fake_images.detach())
    d_loss = (
        trial_loss_fn(real_logits, torch.ones_like(real_logits))
        + trial_loss_fn(fake_logits, torch.zeros_like(fake_logits))
    ) / 2
    assert torch.isfinite(d_loss).item(), "Discriminator loss is not finite."
    d_loss.backward()
    trial_d_optimizer.step()

    trial_d.requires_grad_(False)
    trial_g_optimizer.zero_grad(set_to_none=True)
    fake_logits = trial_d(fake_images)
    g_loss = trial_loss_fn(fake_logits, torch.ones_like(fake_logits))
    assert torch.isfinite(g_loss).item(), "Generator loss is not finite."
    g_loss.backward()
    trial_g_optimizer.step()
    trial_d.requires_grad_(True)

    print(
        f"Batch {step}/{len(trial_loader)} | "
        f"D loss: {d_loss.item():.4f} | G loss: {g_loss.item():.4f}"
    )

print("Training device:", real_images.device)
if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = perf_counter() - start_time
images_per_second = len(trial_dataset) / elapsed
epoch_minutes = len(wikiart_paths) / images_per_second / 60

print(f"Throughput: {images_per_second:.1f} images/sec")
print(f"Estimated time per full epoch: {epoch_minutes:.1f} min")

In [ ]:
from time import perf_counter
import csv

from torchvision.utils import save_image

run_dir.mkdir(parents=True, exist_ok=False)

train_paths = sorted(
    path for path in data_dir.rglob("*")
    if path.suffix.lower() in image_extensions and path.is_file()
)
if not train_paths:
    raise RuntimeError(f"No training images found under {data_dir}")

batch_size = 32
epochs = 1
learning_rate = 0.0002
betas = (0.5, 0.999)

train_dataset = WikiArtDataset(train_paths, image_transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

train_g = Generator(latent_dim, base_channels).to(device)
train_d = Discriminator(base_channels).to(device)
g_optimizer = torch.optim.Adam(
    train_g.parameters(), lr=learning_rate, betas=betas
)
d_optimizer = torch.optim.Adam(
    train_d.parameters(), lr=learning_rate, betas=betas
)
loss_fn = nn.BCEWithLogitsLoss()
fixed_noise = torch.randn(64, latent_dim, 1, 1, device=device)

print(f"Training images: {len(train_dataset):,}", flush=True)
print(f"Batches per epoch: {len(train_loader):,}", flush=True)
print(f"Training device: {next(train_g.parameters()).device}", flush=True)

global_step = 0
with (run_dir / "loss.csv").open("w", newline="") as loss_file:
    writer = csv.writer(loss_file)
    writer.writerow(["epoch", "step", "d_loss", "g_loss"])

    for epoch in range(1, epochs + 1):
        start_time = perf_counter()
        train_g.train()
        train_d.train()

        for batch, real_images in enumerate(train_loader, start=1):
            real_images = real_images.to(device, non_blocking=True)
            noise = torch.randn(
                real_images.size(0), latent_dim, 1, 1, device=device
            )
            fake_images = train_g(noise)

            d_optimizer.zero_grad(set_to_none=True)
            real_logits = train_d(real_images)
            fake_logits = train_d(fake_images.detach())
            d_loss = (
                loss_fn(real_logits, torch.ones_like(real_logits))
                + loss_fn(fake_logits, torch.zeros_like(fake_logits))
            ) / 2
            d_loss.backward()
            d_optimizer.step()

            train_d.requires_grad_(False)
            g_optimizer.zero_grad(set_to_none=True)
            fake_logits = train_d(fake_images)
            g_loss = loss_fn(fake_logits, torch.ones_like(fake_logits))
            g_loss.backward()
            g_optimizer.step()
            train_d.requires_grad_(True)

            global_step += 1
            d_value, g_value = torch.stack(
                [d_loss.detach(), g_loss.detach()]
            ).cpu().tolist()
            writer.writerow([epoch, global_step, d_value, g_value])

            if batch == 1 or batch % 100 == 0 or batch == len(train_loader):
                loss_file.flush()
                print(
                    f"Epoch {epoch}/{epochs} | Batch {batch}/{len(train_loader)} | "
                    f"D: {d_value:.4f} | G: {g_value:.4f}",
                    flush=True,
                )

        train_g.eval()
        with torch.inference_mode():
            samples = (train_g(fixed_noise).cpu() * 0.5 + 0.5).clamp(0, 1)

        save_image(samples, run_dir / f"samples_epoch_{epoch:03d}.png", nrow=8)
        torch.save(
            {
                "epoch": epoch,
                "global_step": global_step,
                "generator": train_g.state_dict(),
                "discriminator": train_d.state_dict(),
                "g_optimizer": g_optimizer.state_dict(),
                "d_optimizer": d_optimizer.state_dict(),
                "fixed_noise": fixed_noise.cpu(),
                "latent_dim": latent_dim,
                "base_channels": base_channels,
                "batch_size": batch_size,
            },
            run_dir / f"checkpoint_epoch_{epoch:03d}.pt",
        )
        print(f"Epoch time: {(perf_counter() - start_time) / 60:.1f} min")

print("Results saved to:", run_dir)

In [ ]:
from IPython.display import Image as DisplayImage, display

display(DisplayImage(filename=str(run_dir / "samples_epoch_001.png")))

In [ ]:
from PIL import Image
from time import perf_counter
import csv

from torchvision.utils import save_image

if not run_dir.is_dir():
    raise FileNotFoundError(run_dir)

train_paths = sorted(
    path for path in data_dir.rglob("*")
    if path.suffix.lower() in image_extensions and path.is_file()
)
if not train_paths:
    raise RuntimeError(f"No training images found under {data_dir}")

batch_size = 32
epochs = 3
learning_rate = 0.0002
betas = (0.5, 0.999)

train_dataset = WikiArtDataset(train_paths, image_transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

train_g = Generator(latent_dim, base_channels).to(device)
train_d = Discriminator(base_channels).to(device)
g_optimizer = torch.optim.Adam(
    train_g.parameters(), lr=learning_rate, betas=betas
)
d_optimizer = torch.optim.Adam(
    train_d.parameters(), lr=learning_rate, betas=betas
)
loss_fn = nn.BCEWithLogitsLoss()
checkpoint = torch.load(
    run_dir / "checkpoint_epoch_001.pt",
    map_location="cpu",
    weights_only=True,
)

train_g.load_state_dict(checkpoint["generator"])
train_d.load_state_dict(checkpoint["discriminator"])
g_optimizer.load_state_dict(checkpoint["g_optimizer"])
d_optimizer.load_state_dict(checkpoint["d_optimizer"])
fixed_noise = checkpoint["fixed_noise"].to(device)

start_epoch = checkpoint["epoch"] + 1
global_step = checkpoint["global_step"]
loss_path = run_dir / "loss.csv"

for epoch in range(start_epoch, epochs + 1):
    if (run_dir / f"checkpoint_epoch_{epoch:03d}.pt").exists():
        raise FileExistsError(f"Checkpoint for epoch {epoch} already exists.")

with loss_path.open("r", newline="") as loss_file:
    last_row = None
    for last_row in csv.DictReader(loss_file):
        pass

if last_row is None or int(last_row["step"]) != global_step:
    raise RuntimeError("The loss log does not end at the checkpoint step.")

print(f"Resuming at epoch {start_epoch}, after step {global_step}.", flush=True)

with loss_path.open("a", newline="") as loss_file:
    writer = csv.writer(loss_file)

    for epoch in range(start_epoch, epochs + 1):

        start_time = perf_counter()
        train_g.train()
        train_d.train()

        for batch, real_images in enumerate(train_loader, start=1):
            real_images = real_images.to(device, non_blocking=True)
            noise = torch.randn(
                real_images.size(0), latent_dim, 1, 1, device=device
            )
            fake_images = train_g(noise)

            d_optimizer.zero_grad(set_to_none=True)
            real_logits = train_d(real_images)
            fake_logits = train_d(fake_images.detach())
            d_loss = (
                loss_fn(real_logits, torch.ones_like(real_logits))
                + loss_fn(fake_logits, torch.zeros_like(fake_logits))
            ) / 2
            d_loss.backward()
            d_optimizer.step()

            train_d.requires_grad_(False)
            g_optimizer.zero_grad(set_to_none=True)
            fake_logits = train_d(fake_images)
            g_loss = loss_fn(fake_logits, torch.ones_like(fake_logits))
            g_loss.backward()
            g_optimizer.step()
            train_d.requires_grad_(True)

            global_step += 1
            d_value, g_value = torch.stack(
                [d_loss.detach(), g_loss.detach()]
            ).cpu().tolist()
            writer.writerow([epoch, global_step, d_value, g_value])

            if batch == 1 or batch % 100 == 0 or batch == len(train_loader):
                loss_file.flush()
                print(
                    f"Epoch {epoch}/{epochs} | Batch {batch}/{len(train_loader)} | "
                    f"D: {d_value:.4f} | G: {g_value:.4f}",
                    flush=True,
                )

        train_g.eval()
        with torch.inference_mode():
            samples = (train_g(fixed_noise).cpu() * 0.5 + 0.5).clamp(0, 1)

        save_image(samples, run_dir / f"samples_epoch_{epoch:03d}.png", nrow=8)
        torch.save(
            {
                "epoch": epoch,
                "global_step": global_step,
                "generator": train_g.state_dict(),
                "discriminator": train_d.state_dict(),
                "g_optimizer": g_optimizer.state_dict(),
                "d_optimizer": d_optimizer.state_dict(),
                "fixed_noise": fixed_noise.cpu(),
                "latent_dim": latent_dim,
                "base_channels": base_channels,
                "batch_size": batch_size,
            },
            run_dir / f"checkpoint_epoch_{epoch:03d}.pt",
        )
        print(f"Epoch time: {(perf_counter() - start_time) / 60:.1f} min")

print("Results saved to:", run_dir)

In [ ]:
from IPython.display import Image as DisplayImage, display

sample_dir = run_dir

for epoch in (1, 2, 3):
    print(f"Epoch {epoch}")
    display(DisplayImage(filename=str(
        sample_dir / f"samples_epoch_{epoch:03d}.png"
    )))

In [ ]:
import csv
from statistics import mean

import matplotlib.pyplot as plt

results_dir = run_dir

with (results_dir / "loss.csv").open(newline="") as loss_file:
    rows = list(csv.DictReader(loss_file))

steps = [int(row["step"]) for row in rows]
epochs = [int(row["epoch"]) for row in rows]
print(f"Logged steps: {len(steps):,} | Last step: {steps[-1]:,}")
print(f"Epochs: {sorted(set(epochs))}")

window = 100
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

for ax, key, label in zip(
    axes,
    ("d_loss", "g_loss"),
    ("Discriminator loss", "Generator loss"),
):
    values = [float(row[key]) for row in rows]
    ax.plot(steps, values, alpha=0.20, linewidth=0.6, label="Per batch")

    ends = list(range(window, len(values) + 1))
    moving_average = [mean(values[end - window:end]) for end in ends]
    ax.plot(
        steps[window - 1:],
        moving_average,
        linewidth=1.4,
        label=f"{window}-step moving average",
    )

    for i in range(1, len(epochs)):
        if epochs[i] != epochs[i - 1]:
            ax.axvline(steps[i] - 0.5, color="gray", linestyle="--", linewidth=0.8)

    ax.set_ylabel(label)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Training step")
fig.tight_layout()
fig.savefig(results_dir / "training_loss_curves.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import torch

eval_dir = run_dir
eval_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    eval_dir / "checkpoint_epoch_003.pt",
    map_location="cpu",
    weights_only=True,
)

eval_generator = Generator(
    checkpoint["latent_dim"], checkpoint["base_channels"]
).to(eval_device)
eval_generator.load_state_dict(checkpoint["generator"])
eval_generator.eval()

sample_count = 5000
eval_batch_size = 64
noise_rng = torch.Generator().manual_seed(285)

generated = torch.empty((sample_count, 3, 64, 64), dtype=torch.uint8)

with torch.inference_mode():
    for start in range(0, sample_count, eval_batch_size):
        end = min(start + eval_batch_size, sample_count)
        noise = torch.randn(
            end - start, checkpoint["latent_dim"], 1, 1, generator=noise_rng
        ).to(eval_device)
        images = eval_generator(noise)
        pixels = ((images + 1) * 127.5).round().clamp(0, 255)
        generated[start:end] = pixels.to(device="cpu", dtype=torch.uint8)

sample_path = eval_dir / "eval_fake_epoch_003.pt"
torch.save(generated, sample_path)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Generated shape:", tuple(generated.shape))
print("Saved to:", sample_path)

In [ ]:
import json
import random

import torch
from PIL import Image as PILImage
from ignite.metrics import FID, InceptionScore
from torchvision.transforms import functional as TF

results_dir = run_dir
wikiart_dir = data_dir
metric_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fake_images = torch.load(
    results_dir / "eval_fake_epoch_003.pt",
    map_location="cpu",
    weights_only=True,
)

extensions = {".jpg", ".jpeg", ".png", ".webp"}
real_paths = sorted(
    path for path in wikiart_dir.rglob("*")
    if path.suffix.lower() in extensions and path.is_file()
)
real_paths = random.Random(285).sample(real_paths, len(fake_images))

fid = FID(device=metric_device)
inception_score = InceptionScore(device=metric_device)
batch_size = 32
print(f"Evaluating {len(fake_images):,} real/fake pairs on {metric_device}.", flush=True)

with torch.inference_mode():
    for batch_index, start in enumerate(range(0, len(fake_images), batch_size), start=1):
        end = min(start + batch_size, len(fake_images))
        real_batch = []
        for path in real_paths[start:end]:
            with PILImage.open(path) as image:
                crop = TF.center_crop(TF.resize(image.convert("RGB"), 64), [64, 64])
                resized = crop.resize((299, 299), PILImage.Resampling.BILINEAR)
                real_batch.append(TF.to_tensor(resized))

        fake_batch = torch.stack([
            TF.to_tensor(TF.to_pil_image(image).resize(
                (299, 299), PILImage.Resampling.BILINEAR
            ))
            for image in fake_images[start:end]
        ]).to(metric_device)
        real_batch = torch.stack(real_batch).to(metric_device)

        fid.update((fake_batch, real_batch))
        inception_score.update(fake_batch)

        if batch_index == 1 or batch_index % 20 == 0 or end == len(fake_images):
            print(f"Processed {end:,}/{len(fake_images):,} pairs.", flush=True)

print("Computing final scores...", flush=True)
scores = {
    "epoch": 3,
    "real_samples": len(real_paths),
    "generated_samples": len(fake_images),
    "fid_ignite": float(fid.compute()),
    "inception_score_ignite": float(inception_score.compute()),
}

score_path = results_dir / "evaluation_epoch_003.json"
score_path.write_text(json.dumps(scores, indent=2), encoding="utf-8")
print(f"FID: {scores['fid_ignite']:.4f}")
print(f"Inception Score: {scores['inception_score_ignite']:.4f}")
print("Saved to:", score_path)